# 반납량을 추가한 대여소별 다음 한 시간 대여 예측

기존 50개 대여소, 기상·유사 기상 피처, MSE와 3개 시간순 fold를 그대로 사용한다. 출발 예측 대상 대여소에 최근 1시간·3시간 동안 도착한 유효 대여 건수를 추가한다. 기존 3-fold에서 고른 설정과 에포크를 고정해 두 입력의 추가 효과를 비교한다. 4분기는 이전 실험에서 이미 열어본 비교 구간이다.

In [1]:
from itertools import product
from pathlib import Path
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "전처리" / "return_features.py").is_file())
DATA_DIR = ROOT / "분석용 데이터"
# 전처리를 다시 실행한 경우 해당 산출물을 우선 사용한다.
PANEL_FILE = ROOT / "전처리" / "station_hour.csv.gz"
if not PANEL_FILE.is_file():
    PANEL_FILE = DATA_DIR / "station_hour.csv.gz"
MODEL_DIR = ROOT / "모델"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED, BATCH, MAX_EPOCHS = 42, 2048, 18
torch.set_num_threads(4)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
plt.rcParams.update({"font.family": "Malgun Gothic" if os.name == "nt" else "sans-serif",
                     "axes.unicode_minus": False, "figure.dpi": 110})
print(f"PyTorch {torch.__version__} / {torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else DEVICE}")

PyTorch 2.5.1+cu121 / NVIDIA GeForce RTX 3060 Ti


In [2]:
LAGS = [f"rentals_lag_{h}h" for h in range(24, 0, -1)]
WEATHER = ["temperature", "humidity", "precipitation", "windspeed"]
WEATHER_COLS = [v + "_lag1h" for v in WEATHER]
HISTORY = ["rentals_lag_168h", "rentals_mean_3h", "rentals_mean_24h", "rentals_mean_168h"]
FLAGS = ["is_weekend", "is_holiday", *[v + "_missing" for v in WEATHER]]
PERIODS = {"hour": 24, "weekday": 7, "month": 12}
FEATURES = ["station_id", *PERIODS, *FLAGS, *WEATHER_COLS, *HISTORY, *LAGS]
cols = ["target_time", "split", "target_rentals", *FEATURES]
pieces = []
for chunk in pd.read_csv(PANEL_FILE, usecols=cols,
                         parse_dates=["target_time"], chunksize=100_000):
    pieces.append(chunk)
frame = pd.concat(pieces, ignore_index=True)
del pieces
assert not frame.duplicated(['target_time','station_id']).any()

## 반납 기록을 두 입력으로 연결

예를 들어 09시 예측에는 같은 대여소에서 08시 이상 09시 미만에 반납된 횟수와 06시 이상 09시 미만에 반납된 횟수를 넣는다. 대여한 곳은 서울 전체에서 허용한다. 이용시간과 이동거리가 모두 양수인 서울 원본 월별 대여이력에서 반납 시각과 도착 번호를 읽는다. 정각 09시의 반납은 10시 예측부터 들어간다.

반납 집계는 대여 시작월 파일을 사용한다. 2023년 이전에 출발한 대여가 시작 경계의 반납 건수에서 빠질 수 있다. 첫 1시간·3시간의 불완전한 창은 결측 표시와 함께 처리한다. 같은 자전거·출발시각의 중복은 한 건으로 정리하고, 출발지가 충돌하거나 가장 늦은 유효 반납 시점의 도착지가 서로 다른 기록은 제외한다. 실제로 관측된 0건은 그대로 둔다.

In [3]:
import sys
sys.path.insert(0, str(ROOT / "전처리"))
from return_features import OUT as RETURN_FILE, build_return_features

# 로컬 재집계본, 공개 저장소에 포함된 집계본 순으로 찾는다.
if not RETURN_FILE.is_file():
    published_returns = DATA_DIR / RETURN_FILE.name
    if published_returns.is_file():
        RETURN_FILE = published_returns
    else:
        build_return_features()
RETURN_COLS = ["returns_prev_1h", "returns_prev_3h"]
returns = pd.read_csv(RETURN_FILE, parse_dates=["target_time"])
keys = ["station_id", "target_time"]
original_keys = frame[keys].copy()
frame = frame.merge(returns, on=keys, how="left", validate="one_to_one", indicator=True)
assert frame.pop("_merge").eq("both").all() and frame[keys].equals(original_keys)
assert len(frame) == len(returns) == 877200
assert frame[RETURN_COLS].isna().sum().tolist() == [50, 150]
display(frame.loc[frame.station_id.eq(221), ["target_time", "station_id", *RETURN_COLS]].head(8))
del original_keys, returns

,target_time,station_id,returns_prev_1h,returns_prev_3h
0,2023-01-01 00:00:00,221,NaN,NaN
1,2023-01-01 01:00:00,221,1.0,NaN
2,2023-01-01 02:00:00,221,1.0,NaN
3,2023-01-01 03:00:00,221,0.0,2.0
4,2023-01-01 04:00:00,221,0.0,1.0
5,2023-01-01 05:00:00,221,0.0,0.0
6,2023-01-01 06:00:00,221,0.0,0.0
7,2023-01-01 07:00:00,221,0.0,0.0


### 같은 대여소·시간대의 유사 기상 사례

이 실험에서는 최근 365일에서 같은 시각과 같은 요일 유형(평일 또는 주말·공휴일)의 날짜를 찾는다. 두 시각 모두 직전 1시간 관측값이 있고 강수 유무가 같아야 한다. 기온 5°C, 습도 15%p, 강수량 2mm, 풍속 2m/s를 변수별 허용 차이로 정한다. 각 차이를 허용 차이로 나눈 뒤 제곱합이 가장 작은 5개까지 선택하고 거리가 같으면 최근 날짜를 우선한다. 이는 이 실험에서 정한 기준이다.

서울 공통 기상을 쓰므로 유사 날짜는 한 번만 찾고, 대여량은 각 대여소에서 따로 구한다. 과거 사례의 출발 시각에서 24시간 뒤까지 반납이 확인된 건수만 평균한다. 검증 중에도 매시간 당시 확인된 기록으로 갱신한다. 유사 사례가 없으면 결측으로 표시하고, 실제 0건과 구분한다.

In [4]:
MATCH_CONFIG = {"window_days": 365, "neighbors": 5,
                "limits": {"temperature": 5.0, "humidity": 15.0, "precipitation": 2.0, "windspeed": 2.0}}
feature = "weather_match_mean"
calendar_cols = ["hour", "is_weekend", "is_holiday", *WEATHER_COLS]
calendar = frame[["target_time", *calendar_cols]].drop_duplicates().set_index("target_time").sort_index()
assert calendar.index.is_unique
times = calendar.index
assert (times[1:] - times[:-1] == pd.Timedelta(hours=1)).all()
weather = calendar[WEATHER_COLS].to_numpy()
limits = np.array([MATCH_CONFIG["limits"][name] for name in WEATHER])
complete = np.isfinite(weather).all(axis=1)
wet = calendar.precipitation_lag1h.to_numpy() > 0
off = (calendar.is_weekend | calendar.is_holiday).to_numpy(dtype=bool)
# 행 t의 24시간 시차 값은 t-24의 출발 중 t까지 반납이 확인된 건수다.
known = frame.pivot(index="target_time", columns="station_id", values="rentals_lag_24h").reindex(times)
assert known.iloc[24:].notna().all().all()
counts = known.to_numpy()
means = np.full(counts.shape, np.nan, dtype=np.float32)
matches = []
day_offsets = 24 * np.arange(1, MATCH_CONFIG["window_days"] + 1)
for t in range(len(times)):
    candidates = t - day_offsets[day_offsets <= t]
    candidates = candidates[complete[candidates] & (off[candidates] == off[t]) & (wet[candidates] == wet[t])]
    if not complete[t]:
        candidates = candidates[:0]
    differences = np.abs(weather[candidates] - weather[t]) / limits
    within = (differences <= 1).all(axis=1)
    distance = (differences[within] ** 2).sum(axis=1)
    nearest = candidates[within][np.argsort(distance, kind="stable")[:MATCH_CONFIG["neighbors"]]]
    matches.append(nearest)
    if nearest.size:
        assert (nearest + 24 <= t).all()
        means[t] = counts[nearest + 24].mean(axis=0)

frame[feature] = means[times.get_indexer(frame.target_time), known.columns.get_indexer(frame.station_id)]
# HISTORY의 다른 시차와 같은 방식으로 log1p와 결측 표시를 모델에 전달한다.
HISTORY = [*HISTORY, feature]


In [5]:
# 과거 건수 입력과 마찬가지로 log1p 변환과 결측 표시를 사용한다.
HISTORY = [*HISTORY, *RETURN_COLS]

### 학습 마감과 입력 변환
모든 학습 시각에서 예측 시점에 쓸 수 있던 시차 피처를 사용한다. 학습 정답에는 해당 시간의 출발 후 24시간까지 확인된 반납 건수를 사용한다. 분기 경계 직전 24시간은 경계 시각까지 실제 확인된 시차별 건수를 쓴다. 검증과 테스트의 정답은 최종 유효 대여 건수다. 날씨 표준화는 fold의 학습 자료에만 맞춘다.

In [6]:
FOLDS = [("1분기", "2024-01-01", "2024-04-01"), ("2분기", "2024-04-01", "2024-07-01"),
         ("3분기", "2024-07-01", "2024-10-01")]
TEST = ("4분기 테스트", "2024-10-01", "2025-01-01")
assert frame.station_id.nunique() == 50 and len(frame) == 877200
asof_24h = frame.sort_values(["station_id", "target_time"]).groupby("station_id")["rentals_lag_24h"].shift(-24)
def split_at(start, end):
    cutoff = pd.Timestamp(start)
    training = frame.loc[frame.target_time < cutoff].copy()
    evaluation = frame.loc[frame.target_time.between(cutoff, pd.Timestamp(end), inclusive="left")]
    labels = asof_24h.loc[training.index].copy()
    boundary = frame.loc[frame.target_time == cutoff].set_index("station_id")
    for h in range(1, 25):
        last = training.target_time == cutoff - pd.Timedelta(hours=h)
        labels.loc[last] = training.loc[last, "station_id"].map(boundary[f"rentals_lag_{h}h"])
    assert labels.notna().all() and labels.le(training.target_rentals).all()
    training["target_rentals"] = labels
    assert len(evaluation) == 50 * 24 * (pd.Timestamp(end) - cutoff).days
    assert training.target_time.max() < evaluation.target_time.min()
    return training, evaluation

In [7]:
def fit_preprocessing(data):
    weather = data[WEATHER_COLS]
    med = weather.median()
    filled = weather.fillna(med)
    return {"stations": sorted(map(int, data.station_id.unique())),
            "median": med.tolist(), "mean": filled.mean().tolist(),
            "std": filled.std(ddof=0).clip(lower=1e-6).tolist()}


def tensors(data, prep, use_weather=True):
    lag = data[LAGS].to_numpy(dtype="float32")
    seq = np.stack([np.log1p(np.nan_to_num(lag)), np.isnan(lag)], axis=-1)
    history_cols = HISTORY if use_weather else [name for name in HISTORY if name != feature]
    hist = data[history_cols].to_numpy(dtype="float32")
    weather = data[WEATHER_COLS].to_numpy(dtype="float32")
    wx = (np.where(np.isnan(weather), prep["median"], weather) - prep["mean"]) / prep["std"]
    calendar = []
    for col, size in PERIODS.items():
        idx = data[col].to_numpy() - int(col == "month")
        calendar.append(np.eye(size, dtype="float32")[idx])
    observed = [data[FLAGS], wx] if use_weather else [data[["is_weekend", "is_holiday"]]]
    context = np.column_stack([*calendar, *observed, np.log1p(np.nan_to_num(hist)), np.isnan(hist)])
    ids = pd.Categorical(data.station_id, categories=prep["stations"]).codes.astype("int64")
    assert ids.min() >= 0 and np.isfinite(context).all() and np.isfinite(seq).all()
    return (torch.as_tensor(seq, dtype=torch.float32, device=DEVICE),
            torch.as_tensor(context, dtype=torch.float32, device=DEVICE),
            torch.as_tensor(ids, dtype=torch.long, device=DEVICE),
            torch.as_tensor(data.target_rentals.to_numpy(), dtype=torch.float32, device=DEVICE))

In [8]:
class DemandNet(nn.Module):
    def __init__(self, kind, context_dim, n_stations, config):
        super().__init__()
        self.kind = kind
        width = config['width']
        self.station = nn.Embedding(n_stations, 8)
        if kind == "GRU":
            self.encoder = nn.GRU(input_size=2, hidden_size=config['gru_hidden'], batch_first=True)
            sequence_dim = config['gru_hidden']
        else:
            self.encoder = nn.Flatten(start_dim=1)
            sequence_dim = 24 * 2
        self.head = nn.Sequential(
            nn.Linear(sequence_dim + context_dim + 8, width), nn.ReLU(), nn.Dropout(config['dropout']),
            nn.Linear(width, width // 2), nn.ReLU(), nn.Linear(width // 2, 1), nn.Softplus())

    def forward(self, seq, context, station):
        if self.kind == "GRU":
            _, hidden = self.encoder(seq)
            encoded = hidden[-1]
        else:
            encoded = self.encoder(seq)
        merged = torch.cat([encoded, context, self.station(station)], dim=1)
        return self.head(merged).squeeze(1) + 1e-6


@torch.inference_mode()
def predict(model, inputs):
    model.eval()
    seq, context, stations, target = inputs
    batches = [model(seq[i:i+BATCH], context[i:i+BATCH], stations[i:i+BATCH])
               for i in range(0, len(target), BATCH)]
    return torch.cat(batches).cpu().numpy()

## 같은 설정에서 반납 입력 추가 전후 비교

학습률 0.001, 은닉층 192·96, 드롭아웃 0.1, 시드 42와 배치 크기 2,048을 고정한다. 기존 교차검증에서 고른 MLP 18에포크와 GRU 15에포크(은닉 크기 48)를 그대로 적용한다. 각 분기의 기상 포함 모델과 동일한 검증 행에서 비교한다.

In [9]:
CHECK_EPOCHS = (5, 10, 15, 18)
def metrics(actual, pred):
    error = np.asarray(actual, dtype="float64") - pred
    assert np.isfinite(error).all()
    return {"MAE": np.abs(error).mean(), "RMSE": np.sqrt(np.mean(error**2))}

def train_network(kind, inputs, n_stations, epochs, config, monitor=None):
    torch.manual_seed(SEED)
    seq, context, stations, target = inputs
    model = DemandNet(kind, context.shape[1], n_stations, config).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=config["lr"])
    loss_fn = nn.MSELoss()
    history = []
    for epoch in range(1, epochs + 1):
        model.train()
        for idx in torch.randperm(len(target), device=DEVICE).split(BATCH):
            optimizer.zero_grad(set_to_none=True)
            mu = model(seq[idx], context[idx], stations[idx])
            loss = loss_fn(mu, target[idx])
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
        if monitor is not None and epoch in CHECK_EPOCHS:
            history.append({"epoch": epoch, **metrics(monitor[-1].cpu().numpy(), predict(model, monitor))})
    return model, history

In [10]:
settings = {
    "MLP": ({"lr": 0.001, "width": 192, "dropout": 0.1}, 18),
    "GRU": ({"lr": 0.001, "width": 192, "dropout": 0.1, "gru_hidden": 48}, 15),
}
reference_cv = pd.read_csv(MODEL_DIR / "3fold 분기별 성능.csv")
reference_test = pd.read_csv(MODEL_DIR / "3fold 최종 테스트.csv").assign(기간=TEST[0])
reference = pd.concat([reference_cv, reference_test])
reference = reference.loc[reference["입력"].eq("기상 포함")].set_index(["기간", "모델"])
rows = []
for period, start, end in [*FOLDS, TEST]:
    training, evaluation = split_at(start, end)
    prep = fit_preprocessing(training)
    inputs, monitor = tensors(training, prep), tensors(evaluation, prep)
    for kind, (config, epochs) in settings.items():
        model, _ = train_network(kind, inputs, len(prep["stations"]), epochs, config)
        scores = metrics(evaluation.target_rentals, predict(model, monitor))
        previous = reference.loc[(period, kind)]
        assert previous["행"] == len(evaluation)
        rows.append({"기간": period, "모델": kind, "행": len(evaluation),
                     "기존 MAE": previous.MAE, "반납 추가 MAE": scores["MAE"],
                     "기존 RMSE": previous.RMSE, "반납 추가 RMSE": scores["RMSE"]})
        del model
    del training, evaluation, inputs, monitor
    print(f"{period} MLP·GRU 완료", flush=True)
result = pd.DataFrame(rows)
result["MAE 차이"] = result["반납 추가 MAE"] - result["기존 MAE"]
result["RMSE 차이"] = result["반납 추가 RMSE"] - result["기존 RMSE"]
display(result.round(4))
display(result.loc[result["기간"] != TEST[0]].groupby("모델")[["기존 MAE", "반납 추가 MAE", "기존 RMSE", "반납 추가 RMSE"]].mean().round(4))
result.to_csv(MODEL_DIR / "3fold 반납 추가 성능.csv", index=False, encoding="utf-8-sig")

1분기 MLP·GRU 완료


2분기 MLP·GRU 완료


3분기 MLP·GRU 완료


4분기 테스트 MLP·GRU 완료


,기간,모델,행,기존 MAE,반납 추가 MAE,기존 RMSE,반납 추가 RMSE,MAE 차이,RMSE 차이
0,1분기,MLP,109200,0.8407,0.8312,1.3380,1.3181,-0.0095,-0.0199
1,1분기,GRU,109200,0.8463,0.8365,1.3483,1.3225,-0.0099,-0.0258
2,2분기,MLP,109200,1.2502,1.2302,1.9862,1.9501,-0.0200,-0.0361
3,2분기,GRU,109200,1.2477,1.2239,2.0119,1.9563,-0.0238,-0.0556
4,3분기,MLP,110400,1.1220,1.0957,1.7805,1.7498,-0.0263,-0.0307
5,3분기,GRU,110400,1.1401,1.1158,1.7808,1.7603,-0.0243,-0.0206
6,4분기 테스트,MLP,110400,0.9819,0.9491,1.5567,1.5246,-0.0328,-0.0321
7,4분기 테스트,GRU,110400,0.9691,0.9501,1.5538,1.5297,-0.0190,-0.0241


,기존 MAE,반납 추가 MAE,기존 RMSE,반납 추가 RMSE
모델,,,,
GRU,1.078,1.0587,1.7137,1.6797
MLP,1.071,1.0524,1.7016,1.6727
